This notebook evaluates a simple static baseline using logistic regression on mean-pooled clip-level pose features. It is intended as a comparison model against the primary temporal GRU.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.dataset import load_processed_metadata
from src.train_baselines import (
    build_static_feature_matrix,
    run_logistic_regression_cv,
    BASELINE_RESULTS_PATH,
    BASELINE_PREDICTIONS_PATH,
    BASELINE_SUMMARY_PATH,
    BASELINE_CONFUSION_MATRIX_PATH,
)
from src.config import INDEX_TO_LABEL

In [2]:
processed_meta = load_processed_metadata()

print("Number of processed clips:", len(processed_meta))
print(processed_meta["label"].value_counts())
processed_meta.head()

Number of processed clips: 30
label
bad_jump     15
good_jump    15
Name: count, dtype: int64


,clip_id,video_path,label,sequence_path,num_sampled_frames,missing_pose_frames,processed_shape
0,bad_jump_01,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
1,bad_jump_02,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
2,bad_jump_03,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"
3,bad_jump_04,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,1,"(30, 132)"
4,bad_jump_05,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,bad_jump,/Users/arjunrammohandas/Desktop/JumpSafe_Conti...,30,0,"(30, 132)"


In [3]:
X, y, clip_ids = build_static_feature_matrix(processed_meta)

print("Feature matrix shape:", X.shape)
print("Label shape:", y.shape)
print("First clip ID:", clip_ids[0])

Feature matrix shape: (30, 132)
Label shape: (30,)
First clip ID: bad_jump_01


In [4]:
baseline_outputs = run_logistic_regression_cv(
    metadata_df=processed_meta,
    n_splits=5,
    save_results=True,
)

In [5]:
baseline_outputs["fold_results"]

,fold,accuracy,precision,recall,f1
0,1,0.666667,1.000000,0.333333,0.500000
1,2,0.666667,0.600000,1.000000,0.750000
2,3,0.666667,0.666667,0.666667,0.666667
3,4,0.500000,0.500000,0.333333,0.400000
4,5,0.666667,0.600000,1.000000,0.750000


In [6]:
baseline_outputs["summary"]

,metric,mean,std
0,accuracy,0.633333,0.074536
1,precision,0.673333,0.192065
2,recall,0.666667,0.333333
3,f1,0.613333,0.156968


In [7]:
baseline_outputs["confusion_matrix"]

,pred_0,pred_1
true_0,9,6
true_1,5,10


In [8]:
pred_df = baseline_outputs["predictions"].copy()
pred_df["true_label_name"] = pred_df["true_label"].map(INDEX_TO_LABEL)
pred_df["pred_label_name"] = pred_df["pred_label"].map(INDEX_TO_LABEL)

pred_df.head(10)

,fold,clip_id,true_label,pred_label,correct,true_label_name,pred_label_name
0,1,bad_jump_03,0,0,1,bad_jump,bad_jump
1,1,bad_jump_07,0,0,1,bad_jump,bad_jump
2,1,bad_jump_08,0,0,1,bad_jump,bad_jump
3,1,good_jump_03,1,0,0,good_jump,bad_jump
4,1,good_jump_10,1,1,1,good_jump,good_jump
5,1,good_jump_13,1,0,0,good_jump,bad_jump
6,2,bad_jump_05,0,1,0,bad_jump,good_jump
7,2,bad_jump_10,0,0,1,bad_jump,bad_jump
8,2,bad_jump_14,0,1,0,bad_jump,good_jump
9,2,good_jump_08,1,1,1,good_jump,good_jump


In [9]:
pred_df[pred_df["correct"] == 0]

,fold,clip_id,true_label,pred_label,correct,true_label_name,pred_label_name
3,1,good_jump_03,1,0,0,good_jump,bad_jump
5,1,good_jump_13,1,0,0,good_jump,bad_jump
6,2,bad_jump_05,0,1,0,bad_jump,good_jump
8,2,bad_jump_14,0,1,0,bad_jump,good_jump
13,3,bad_jump_11,0,1,0,bad_jump,good_jump
15,3,good_jump_04,1,0,0,good_jump,bad_jump
18,4,bad_jump_01,0,1,0,bad_jump,good_jump
21,4,good_jump_01,1,0,0,good_jump,bad_jump
22,4,good_jump_05,1,0,0,good_jump,bad_jump
24,5,bad_jump_04,0,1,0,bad_jump,good_jump


In [10]:
print(BASELINE_RESULTS_PATH, BASELINE_RESULTS_PATH.exists())
print(BASELINE_PREDICTIONS_PATH, BASELINE_PREDICTIONS_PATH.exists())
print(BASELINE_SUMMARY_PATH, BASELINE_SUMMARY_PATH.exists())
print(BASELINE_CONFUSION_MATRIX_PATH, BASELINE_CONFUSION_MATRIX_PATH.exists())

/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/logreg_cv_results.csv True
/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/logreg_cv_predictions.csv True
/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/logreg_cv_summary.csv True
/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/logreg_cv_confusion_matrix.npy True


Preliminary comparison: under the current settings, logistic regression on aggregated features outperforms the GRU sequence model. This suggests that the present temporal model may require further tuning or more data to realize an advantage.

In [11]:
from src.train_baselines import train_final_logistic_regression_model, BASELINE_MODEL_PATH

final_logreg_model = train_final_logistic_regression_model(processed_meta, save_model=True)

print(BASELINE_MODEL_PATH)
print(BASELINE_MODEL_PATH.exists())

/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/logreg_model.joblib
True


In [12]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.inference import predict_landing_quality_from_video
from src.train_baselines import BASELINE_MODEL_PATH

print(BASELINE_MODEL_PATH)
print(BASELINE_MODEL_PATH.exists())

/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/processed/logreg_model.joblib
True


In [13]:
sample_video = PROJECT_ROOT / "data" / "raw" / "bad" / "bad_jump_01.MOV"
print(sample_video)
print(sample_video.exists())

/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/raw/bad/bad_jump_01.MOV
True


In [14]:
result = predict_landing_quality_from_video(sample_video)
result

I0000 00:00:1775860830.063715 39083279 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1775860830.177479 39083626 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775860830.205326 39083625 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1775860830.223898 39083627 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


{'video_path': '/Users/arjunrammohandas/Desktop/JumpSafe_Continuation/data/raw/bad/bad_jump_01.MOV',
 'predicted_label': 'bad_jump',
 'predicted_index': 0,
 'confidence': 0.8572569954274166,
 'probabilities': {'bad_jump': 0.8572569954274166,
  'good_jump': 0.14274300457258338},
 'num_sampled_frames': 30,
 'missing_pose_frames': 0,
 'processed_sequence_shape': (30, 132)}